# Custom MCP for an internal API

Most internal APIs weren't designed for LLMs. Wrapping them in an MCP server gives you a chance to design the **agent-facing** surface — fewer tools, friendlier types, opinionated defaults — while leaving the upstream service unchanged.

This notebook wraps a simulated `internal-weather` API. Notice three things in the wrap:
* The MCP tool exposes 2 parameters; the upstream payload has ~15 fields.
* Verbose enums (`PARTLY_CLOUDY`) become human-readable strings.
* `set_alert` is destructive, so the wrap layer adds a rate limit.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '06-mcp'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

# Re-use the eval module's server builder so the notebook stays small.
import importlib.util
leaf = ROOT / '06-mcp' / '03-custom-mcp-for-internal-api' / 'eval.py'
spec = importlib.util.spec_from_file_location('weather_eval', leaf)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

from mcp_core import Client, InProcessTransport
server, rate, upstream_bytes = mod.build_weather_server()
client = Client(transport=InProcessTransport(server)); client.initialize()
[t['name'] for t in client.list_tools()]

In [ ]:
import json
from mcp_core import search_corpus  # demonstrates same Client surface

# Upstream payload (verbose):
raw = mod.UPSTREAM_FIXTURES['Bengaluru']
print('upstream bytes :', len(json.dumps(raw)))

# Wrapped MCP call (trim + humanize):
wrapped = client.call_tool('get_forecast', {'city': 'Bengaluru'})
print('wrapped bytes  :', len(json.dumps(wrapped)))
wrapped

In [ ]:
# Rate limit demo: 3 alerts, max 2 per window -> the 3rd should be rejected.
for city, thr in [('Bengaluru', 35.0), ('Seattle', 25.0), ('Bengaluru', 36.0)]:
    print(city, '->', client.call_tool('set_alert', {'city': city, 'threshold_c': thr}))

## Why this design wins

* The LLM sees a small, opinionated surface (`city`, `units`, `threshold_c`) — easier to prompt, easier to evaluate.
* Cross-cutting concerns (rate limits, retries, output trimming) live in the wrap, not the underlying service.
* When the upstream API changes, you update the wrap; the agent prompt is unchanged.